# Cosmic Expansion from Lattice Dynamics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gpartin/LFMPublicExperiments/blob/main/LFMPublicExperiments/notebooks/LFM_Cosmic_Expansion.ipynb)

## What This Notebook Demonstrates

A wave packet propagating from a **low-$\chi$** region (matter-rich, early universe) into a **high-$\chi$** region (matter-poor, late universe) undergoes **wavelength stretching** \u2014 cosmological **redshift**.

The mechanism: the KG dispersion relation $\omega^2 = c^2 k^2 + \chi^2$ conserves $\omega$ during propagation, so as $\chi$ increases, $k$ must decrease $\Rightarrow$ wavelength increases.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

C = 1.0

def laplacian_1d(f, dx):
    lap = np.zeros_like(f)
    lap[1:-1] = (f[2:] + f[:-2] - 2*f[1:-1]) / dx**2
    return lap

# Grid: fine resolution to resolve short wavelengths
N = 2000
dx = 0.2
dt = 0.05
x = np.arange(N) * dx  # 0 to 400
center_x = N * dx / 2   # x = 200

# Chi profile: smooth transition from low to high
# Low chi (left) = matter-rich (early universe)
# High chi (right) = matter-poor (late universe)
CHI_LEFT = 1.0
CHI_RIGHT = 5.0
chi = CHI_LEFT + (CHI_RIGHT - CHI_LEFT) * 0.5 * (1 + np.tanh((x - center_x) / 15))

# Wave packet with carrier wave, starting in low-chi (left) region
k0 = 6.0     # carrier wavenumber
sigma = 12.0  # envelope width
x0 = 50.0    # start position (well in left region)
omega0 = np.sqrt(C**2 * k0**2 + CHI_LEFT**2)

# Theoretical prediction for right region
k_right = np.sqrt(omega0**2 - CHI_RIGHT**2) / C
lambda_left = 2 * np.pi / k0
lambda_right = 2 * np.pi / k_right
z_theory = lambda_right / lambda_left - 1

# Initialize: propagating rightward
E = np.exp(-(x - x0)**2 / (2 * sigma**2)) * np.cos(k0 * x)
E_prev = np.exp(-(x - x0 + C*k0/omega0*dt)**2 / (2 * sigma**2)) * np.cos(k0 * (x - C*k0/omega0*dt))

n_steps = 6000
snap_steps = [0, 2000, 4000, 5999]
snapshots = {}

for step in range(n_steps):
    if step in snap_steps:
        snapshots[step] = E.copy()
    E_next = 2*E - E_prev + dt**2 * (C**2 * laplacian_1d(E, dx) - chi**2 * E)
    E_prev, E = E, E_next

# Measure wavelength via zero crossings
def measure_wl(E_arr, x_arr, x_lo, x_hi):
    mask = (x_arr >= x_lo) & (x_arr < x_hi)
    region = E_arr[mask]
    crossings = np.where(np.diff(np.sign(region)))[0]
    if len(crossings) >= 4:
        return 2 * np.mean(np.diff(crossings)) * dx
    return None

wl_meas_left = measure_wl(snapshots[0], x, 30, 80)
wl_meas_right = measure_wl(snapshots[5999], x, center_x + 30, center_x + 100)

print("=" * 60)
print("COSMOLOGICAL REDSHIFT FROM chi GRADIENT")
print("=" * 60)
print(f"  Chi left (matter-rich):  {CHI_LEFT}")
print(f"  Chi right (matter-poor): {CHI_RIGHT}")
print(f"  Carrier wavenumber k0:   {k0}")
print(f"  omega (conserved):       {omega0:.3f}")
print(f"\n  Theoretical:")
print(f"    lambda_left  = {lambda_left:.4f}")
print(f"    lambda_right = {lambda_right:.4f}")
print(f"    k_right      = {k_right:.4f}")
print(f"    Redshift z   = {z_theory:.4f}")
print(f"\n  Measured from simulation:")
if wl_meas_left:
    print(f"    lambda_left  = {wl_meas_left:.4f}")
if wl_meas_right:
    print(f"    lambda_right = {wl_meas_right:.4f}")
    z_meas = wl_meas_right / wl_meas_left - 1 if wl_meas_left else None
    if z_meas is not None:
        print(f"    Redshift z   = {z_meas:.4f}")
        print(f"\n  COSMOLOGICAL REDSHIFT CONFIRMED!")
        print(f"  Wave stretches as it enters matter-poor region.")
        print(f"  Mechanism: KG dispersion conserves omega, k adjusts to chi.")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors = ['blue', 'green', 'orange', 'red']
labels = ['Start (low chi)', 'Propagating', 'At transition', 'In high-chi region']
for i, (step, E_snap) in enumerate(sorted(snapshots.items())):
    ax = axes[i // 2, i % 2]
    ax.plot(x, E_snap, color=colors[i], linewidth=0.8)
    ax.fill_between(x, 0, chi / chi.max() * np.max(np.abs(E_snap)) * 0.3,
                    alpha=0.15, color='gray', label='chi profile (scaled)')
    ax.set_title(f'{labels[i]} (step {step})')
    ax.set_xlabel('x')
    ax.set_ylabel('E(x)')
    ax.set_xlim(0, N * dx)
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(alpha=0.3)

plt.suptitle(f'Cosmological Redshift: wavelength stretches from {lambda_left:.3f} to {lambda_right:.3f} (z = {z_theory:.2f})',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Key Result

- A wave packet propagating from low-$\chi$ to high-$\chi$ undergoes **wavelength stretching**
- The KG dispersion $\omega^2 = c^2 k^2 + \chi^2$ conserves $\omega$, forcing $k$ to decrease $\Rightarrow$ $\lambda$ increases
- This is exactly **cosmological redshift** with $\chi$ playing the role of the scale factor
- No FLRW metric was injected \u2014 redshift emerges from substrate dispersion